# LA Studio Subtitle OCR — PP-OCRv5 Multilingual 3.1

This direct CUDA notebook is independent of API Gateway. It loads the
PP-OCRv5 multilingual family (PaddleOCR 3.1.1, Apache-2.0) and accepts
only cropped PNG subtitle frames, never a source video.

1. Choose **Runtime → Change runtime type → GPU**.
2. Run all cells.
3. In Subtitle OCR choose **Colab GPU**, open Configure / check Colab,
   and paste only the temporary URL and token printed below.


In [ ]:
!nvidia-smi
# Keep OCR out of Colab's mutable global site-packages. A
# global install can leave Pillow 12's ImageText.py beside an
# older PIL._typing.py, which causes the `_Ink` error reported
# in this notebook. Do not create a venv: recent Colab Python
# images can have a broken ensurepip bootstrap.
import os
import shutil
import subprocess
import sys
from pathlib import Path

OCR_SITE_PACKAGES = Path("/content/la_studio_subtitle_ocr_site")
shutil.rmtree(OCR_SITE_PACKAGES, ignore_errors=True)
OCR_SITE_PACKAGES.mkdir(parents=True, exist_ok=True)
BOOTSTRAP_ENV = os.environ.copy()
BOOTSTRAP_ENV.pop("PYTHONPATH", None)
BOOTSTRAP_ENV["PYTHONNOUSERSITE"] = "1"

def bootstrap_pip(*arguments):
    subprocess.check_call([sys.executable, "-m", "pip", *arguments], env=BOOTSTRAP_ENV)

def ocr_pip(*arguments):
    bootstrap_pip("install", "--target", str(OCR_SITE_PACKAGES), *arguments)

OCR_PYTHON = sys.executable
OCR_ENV = os.environ.copy()
OCR_ENV["PYTHONPATH"] = str(OCR_SITE_PACKAGES)
OCR_ENV["PYTHONNOUSERSITE"] = "1"
# PaddleOCR 3.1.1 only declares a lower bound for PaddleX. A
# later PaddleX release imports ModelScope/Torch, so retain the
# known 3.1.0 trio inside the dedicated package directory.
ocr_pip("install", "--no-cache-dir", "--upgrade", "--force-reinstall",
        "paddlepaddle-gpu==3.1.0",
        "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu118/")
ocr_pip("install", "--no-cache-dir", "--upgrade", "--force-reinstall",
        "paddleocr==3.1.1", "paddlex[ie,multimodal,ocr,trans]==3.1.0",
        "fastapi==0.115.12", "uvicorn==0.34.3", "python-multipart==0.0.20")
ocr_pip("install", "--no-cache-dir", "--upgrade", "--force-reinstall",
        "--only-binary=:all:", "Pillow==12.0.0")


In [ ]:
# Fail in this explicit dependency probe rather than after the
# service has started. This uses the same interpreter plus
# package-directory environment that will launch Uvicorn.
from textwrap import dedent

probe = dedent(r'''
from importlib.metadata import version
import os
import sys
from pathlib import Path
from PIL import ImageText
from PIL._typing import _Ink
import paddle
from paddleocr import PaddleOCR

assert version("paddlepaddle-gpu") == "3.1.0", version("paddlepaddle-gpu")
assert version("paddlex") == "3.1.0", version("paddlex")
assert version("paddleocr") == "3.1.1", version("paddleocr")
assert version("pillow") == "12.0.0", version("pillow")
assert str(Path(paddle.__file__).resolve()).startswith(str(Path(os.environ["LA_STUDIO_OCR_SITE"]).resolve())), "Paddle must be imported from the dedicated OCR package directory."
assert paddle.device.is_compiled_with_cuda(), "Choose a Colab GPU runtime; CPU fallback is disabled."
paddle.device.set_device("gpu:0")
print("Verified isolated OCR stack:", paddle.__version__, version("paddlex"), version("paddleocr"), version("pillow"))
'''
)
OCR_ENV["LA_STUDIO_OCR_SITE"] = str(OCR_SITE_PACKAGES)
subprocess.run([OCR_PYTHON, "-c", probe], check=True, env=OCR_ENV)


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_subtitle_ocr_worker.py')
WORKER.write_text('import json\nimport os\nimport secrets\nimport tempfile\nimport threading\nfrom pathlib import Path\n\nimport paddle\nfrom fastapi import Depends, FastAPI, File, Form, Header, HTTPException, Request, UploadFile\nfrom fastapi.responses import JSONResponse\nfrom paddleocr import PaddleOCR\n\nif not paddle.device.is_compiled_with_cuda():\n    raise RuntimeError("CUDA is unavailable. Choose a Colab GPU runtime; CPU fallback is disabled.")\npaddle.device.set_device("gpu:0")\nGPU_NAME = paddle.device.cuda.get_device_name(0)\n\nMODEL_ID = "pp-ocrv5-multilingual-3.1"\nMODEL_NAME = "PP-OCRv5 Multilingual 3.1"\nUPSTREAM_MODEL = "PaddlePaddle/PaddleOCR PP-OCRv5"\nUPSTREAM_VERSION = "PaddleOCR 3.1.1"\nLICENSE = "Apache-2.0"\nWORKER_REVISION = "subtitle-ocr-2026-08-11.6"\nRESPONSE_CONTRACT = "subtitle-ocr-crops-v1"\nTOKEN = os.environ["LA_STUDIO_COLAB_SUBTITLE_OCR_TOKEN"]\nMAX_UPLOAD_BYTES = 16 * 1024 * 1024\nINFERENCE_SLOTS = threading.BoundedSemaphore(1)\nENGINE_LOCK = threading.Lock()\nENGINES: dict[str, PaddleOCR] = {}\n\n# The desktop has stored legacy Tesseract codes since Subtitle OCR started.\n# Map them explicitly to the PP-OCRv5 language profiles; unsupported codes are\n# rejected instead of routed to an arbitrary model or a CPU fallback.\nLANGUAGE_PROFILES = {\n    "eng": "en", "en": "en",\n    "vie": "vi", "vi": "vi",\n    "chi_sim": "ch", "chi_tra": "chinese_cht", "ch": "ch", "zh": "ch",\n    "jpn": "japan", "ja": "japan",\n    "kor": "korean", "ko": "korean",\n}\n\n\ndef authorize(authorization: str = Header(default="")):\n    if not secrets.compare_digest(authorization, "Bearer " + TOKEN):\n        raise HTTPException(status_code=401, detail="invalid or missing bearer token")\n\n\ndef require_exact_model(requested: str) -> None:\n    if requested.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{requested}\'. Open the matching notebook.",\n        )\n\n\ndef resolve_profile(language: str) -> str:\n    # A composite Tesseract value (for example eng+chi_sim) is intentionally\n    # rejected. The user must choose one visible OCR language per run, which\n    # makes the actual PP-OCR profile and expected quality unambiguous.\n    normalized = language.strip().lower()\n    if normalized not in LANGUAGE_PROFILES:\n        raise HTTPException(status_code=422, detail=f"unsupported Subtitle OCR language: {language}")\n    return LANGUAGE_PROFILES[normalized]\n\n\ndef engine_for(profile: str) -> PaddleOCR:\n    with ENGINE_LOCK:\n        engine = ENGINES.get(profile)\n        if engine is None:\n            engine = PaddleOCR(\n                lang=profile,\n                ocr_version="PP-OCRv5",\n                device="gpu:0",\n                use_doc_orientation_classify=False,\n                use_doc_unwarping=False,\n                use_textline_orientation=False,\n            )\n            ENGINES[profile] = engine\n        return engine\n\n\ndef result_fields(result: object) -> tuple[list[str], list[float]]:\n    # PaddleOCR 3.x result objects expose a JSON-compatible payload. Keep this\n    # adapter tolerant of the documented result wrappers, but never invent text\n    # when the exact model detected none in a sampled subtitle crop.\n    payload = result\n    if hasattr(payload, "json"):\n        payload = payload.json\n        if callable(payload):\n            payload = payload()\n    if isinstance(payload, str):\n        payload = json.loads(payload)\n    if not isinstance(payload, dict):\n        return [], []\n    data = payload.get("res", payload)\n    if not isinstance(data, dict):\n        return [], []\n    texts = data.get("rec_texts", data.get("text", []))\n    scores = data.get("rec_scores", data.get("scores", []))\n    if isinstance(texts, str):\n        texts = [texts]\n    if not isinstance(texts, list):\n        texts = []\n    if not isinstance(scores, list):\n        scores = []\n    clean_texts = [str(value).strip() for value in texts if str(value).strip()]\n    clean_scores = [float(value) for value in scores[:len(clean_texts)] if isinstance(value, (int, float))]\n    return clean_texts, clean_scores\n\n\napp = FastAPI(title=f"LA Studio Subtitle OCR - {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n\n@app.exception_handler(Exception)\nasync def unhandled_exception(request: Request, error: Exception):\n    detail = f"Subtitle OCR worker internal error: {type(error).__name__}: {str(error)[:300]}"\n    print(detail, flush=True)\n    return JSONResponse(status_code=500, content={"detail": detail})\n\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(_: None = Depends(authorize)):\n    return {\n        "ready": True,\n        "device": "cuda",\n        "gpu": GPU_NAME,\n        "model": MODEL_ID,\n        "variant": "fixed",\n        "upstream_model": UPSTREAM_MODEL,\n        "upstream_version": UPSTREAM_VERSION,\n        "license": LICENSE,\n        "worker_revision": WORKER_REVISION,\n        "response_contract": RESPONSE_CONTRACT,\n        "cpu_fallback": False,\n    }\n\n\n@app.get("/v1/capabilities")\ndef capabilities(_: None = Depends(authorize)):\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "worker_revision": WORKER_REVISION,\n        "capabilities": [{\n            "id": "subtitle-ocr",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "variant": "fixed",\n                "upstream_model": UPSTREAM_MODEL,\n                "upstream_version": UPSTREAM_VERSION,\n                "license": LICENSE,\n                "languages": ["vi", "ch", "japan", "korean", "en"],\n                "device": "cuda",\n                "loaded": True,\n                "response_contract": RESPONSE_CONTRACT,\n            }],\n        }],\n    }\n\n\n@app.post("/v1/ocr/subtitles")\nasync def recognize_subtitle(\n    model: str = Form(...),\n    language: str = Form(...),\n    file: UploadFile = File(...),\n    _: None = Depends(authorize),\n):\n    require_exact_model(model)\n    profile = resolve_profile(language)\n    if file.content_type not in {"image/png", "application/octet-stream"}:\n        raise HTTPException(status_code=415, detail="Subtitle OCR accepts only PNG crop frames")\n    data = await file.read(MAX_UPLOAD_BYTES + 1)\n    if not data or len(data) > MAX_UPLOAD_BYTES or not data.startswith(b"\\x89PNG\\r\\n\\x1a\\n"):\n        raise HTTPException(status_code=400, detail="Subtitle OCR crop must be a non-empty PNG no larger than 16 MiB")\n    if not INFERENCE_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="worker is busy; retry shortly")\n    path = None\n    try:\n        with tempfile.NamedTemporaryFile(prefix="la-studio-subtitle-", suffix=".png", delete=False) as handle:\n            handle.write(data)\n            path = Path(handle.name)\n        texts: list[str] = []\n        scores: list[float] = []\n        for result in engine_for(profile).predict(str(path)):\n            current_texts, current_scores = result_fields(result)\n            texts.extend(current_texts)\n            scores.extend(current_scores)\n        text = " ".join(texts).strip()\n        confidence = sum(scores) / len(scores) if scores else 0.0\n        return {"text": text, "confidence": max(0.0, min(1.0, confidence))}\n    except HTTPException:\n        raise\n    except Exception as error:\n        raise HTTPException(status_code=503, detail=f"PP-OCRv5 inference failed: {type(error).__name__}: {str(error)[:300]}") from error\n    finally:\n        if path is not None:\n            path.unlink(missing_ok=True)\n        INFERENCE_SLOTS.release()' + '\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
MODEL_ID = 'pp-ocrv5-multilingual-3.1'
# LA Studio worker launch contract: launch-2026-08-06.1
import json
import os
import queue
import re
import secrets
import signal
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CAPABILITY_LABEL = 'Subtitle OCR'
MODEL_ID = 'pp-ocrv5-multilingual-3.1'
PORT = 3955
TOKEN_ENV = 'LA_STUDIO_COLAB_SUBTITLE_OCR_TOKEN'
URL_ENV = 'LA_STUDIO_COLAB_SUBTITLE_OCR_URL'
MODEL_ENV = 'LA_STUDIO_COLAB_SUBTITLE_OCR_MODEL'
WORKER_LOG = Path('/content/la_studio_subtitle_ocr_worker.log')
WORKER_MODULE = 'la_studio_subtitle_ocr_worker'
WORKER_PYTHON = sys.executable
WORKER_PYTHON_ISOLATED = False
WORKER_ENVIRONMENT = {"PYTHONNOUSERSITE": "1", "PYTHONPATH": "/content/la_studio_subtitle_ocr_site"}
STARTUP_TIMEOUT_SECONDS = 20 * 60
TUNNEL_TIMEOUT_SECONDS = 90
TOKEN = secrets.token_urlsafe(32)


def port_is_occupied(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.5):
            return True
    except OSError:
        return False


def process_cmdline(pid: int) -> str:
    """Read a Linux process command line without depending on psutil."""
    try:
        return Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode(
            "utf-8", errors="replace"
        ).strip()
    except (FileNotFoundError, PermissionError, ProcessLookupError):
        return ""


def all_processes():
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        pid = int(entry.name)
        command = process_cmdline(pid)
        if command:
            yield pid, command


def listening_processes(port: int) -> dict[int, str]:
    """Return PIDs listening on a local TCP port via /proc socket ownership."""
    target_port = f"{port:04X}"
    socket_inodes = set()
    for table_name in ("/proc/net/tcp", "/proc/net/tcp6"):
        try:
            lines = Path(table_name).read_text(encoding="utf-8").splitlines()[1:]
        except FileNotFoundError:
            continue
        for line in lines:
            fields = line.split()
            if len(fields) < 10:
                continue
            local_address, state, inode = fields[1], fields[3], fields[9]
            if state == "0A" and local_address.rsplit(":", 1)[-1].upper() == target_port:
                socket_inodes.add(inode)
    if not socket_inodes:
        return {}

    listeners = {}
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        try:
            descriptors = (entry / "fd").iterdir()
        except (FileNotFoundError, PermissionError):
            continue
        for descriptor in descriptors:
            try:
                target = os.readlink(descriptor)
            except (FileNotFoundError, PermissionError, OSError):
                continue
            match = re.fullmatch(r"socket:\\[(\\d+)\\]", target)
            if match and match.group(1) in socket_inodes:
                pid = int(entry.name)
                listeners[pid] = process_cmdline(pid)
                break
    return listeners


def stop_pid(pid: int) -> None:
    if pid == os.getpid():
        return
    try:
        os.kill(pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    deadline = time.monotonic() + 10
    while time.monotonic() < deadline:
        try:
            os.kill(pid, 0)
        except ProcessLookupError:
            return
        time.sleep(0.2)
    try:
        os.kill(pid, signal.SIGKILL)
    except ProcessLookupError:
        pass


def reclaim_previous_la_studio_worker() -> None:
    """Stop only an older LA Studio worker/tunnel for this exact local port.

    Re-running a Colab cell keeps child processes alive.  The previous launch
    created a new token but aborted before it could replace the old worker,
    forcing users to destroy the whole GPU runtime.  We identify ownership by
    the exact generated module name and never terminate a foreign listener.
    """
    stopped = []
    for pid, command in listening_processes(PORT).items():
        if WORKER_MODULE in command and "uvicorn" in command:
            stop_pid(pid)
            stopped.append(f"worker PID {pid}")

    endpoint = f"http://127.0.0.1:{PORT}"
    for pid, command in all_processes():
        if ("cloudflared" in command and "tunnel" in command and endpoint in command):
            stop_pid(pid)
            stopped.append(f"tunnel PID {pid}")

    deadline = time.monotonic() + 12
    while port_is_occupied(PORT) and time.monotonic() < deadline:
        time.sleep(0.2)
    if stopped:
        print("Stopped previous LA Studio " + ", ".join(stopped) + ".")

    if port_is_occupied(PORT):
        listeners = listening_processes(PORT)
        foreign_pids = sorted(listeners) or ["unknown"]
        raise RuntimeError(
            f"Port {PORT} is occupied by a process that is not the previous LA Studio "
            f"{CAPABILITY_LABEL} worker (PID(s): {', '.join(map(str, foreign_pids))}). "
            "Choose a fresh Colab runtime rather than terminating an unrelated process."
        )


def worker_log_tail() -> str:
    try:
        return WORKER_LOG.read_text(encoding="utf-8", errors="replace")[-12000:]
    except FileNotFoundError:
        return "(worker log was not created)"


def stop_process(process) -> None:
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()


reclaim_previous_la_studio_worker()

env = os.environ.copy()
env[TOKEN_ENV] = TOKEN
env["PYTHONUNBUFFERED"] = "1"
env.update(WORKER_ENVIRONMENT)
if WORKER_PYTHON_ISOLATED:
    # Do not let Colab's global site-packages or a notebook-level PYTHONPATH
    # bleed into a dedicated worker virtual environment.
    env.pop("PYTHONPATH", None)
    env["PYTHONNOUSERSITE"] = "1"
worker = None
tunnel = None

with WORKER_LOG.open("w", encoding="utf-8", buffering=1) as worker_output:
    worker = subprocess.Popen(
        [WORKER_PYTHON, "-m", "uvicorn", 'la_studio_subtitle_ocr_worker:app', "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content",
        env=env,
        stdout=worker_output,
        stderr=subprocess.STDOUT,
    )
    print(f"Starting exact CUDA {CAPABILITY_LABEL} worker; initial model load can take several minutes.")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS
    last_error = "worker has not answered /health yet"
    next_report = time.monotonic()
    while time.monotonic() < deadline:
        exit_code = worker.poll()
        if exit_code is not None:
            raise RuntimeError(
                f"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\n\n"
                "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
            )
        try:
            request = urllib.request.Request(
                f"http://127.0.0.1:{PORT}/health",
                headers={"Authorization": "Bearer " + TOKEN},
            )
            with urllib.request.urlopen(request, timeout=10) as response:
                health = json.loads(response.read().decode("utf-8"))
            if (response.status == 200
                    and health.get("ready") is True
                    and str(health.get("device", "")).lower() == "cuda"
                    and str(health.get("model", "")).strip().lower() == MODEL_ID
                    and health.get("cpu_fallback") is False):
                print("Exact CUDA worker is ready:", health)
                break
            last_error = "unexpected /health response: " + json.dumps(health, ensure_ascii=False)
        except urllib.error.HTTPError as error:
            last_error = f"/health returned HTTP {error.code}: " + error.read().decode("utf-8", errors="replace")[:1000]
        except Exception as error:
            last_error = f"/health is not ready: {type(error).__name__}: {error}"
        if time.monotonic() >= next_report:
            print("Waiting for the exact CUDA model…", last_error)
            next_report = time.monotonic() + 30
        time.sleep(2)
    else:
        stop_process(worker)
        raise RuntimeError(
            f"The exact-model {CAPABILITY_LABEL} worker did not become CUDA-ready within "
            f"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\n\n"
            "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
        )


def cloudflared_ready() -> bool:
    try:
        return subprocess.run(
            ["cloudflared", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    except OSError:
        return False


def ensure_cloudflared() -> None:
    if cloudflared_ready():
        return
    package_path = "/content/la-studio-cloudflared.deb"
    download = subprocess.run(
        [
            "curl", "--fail", "--location", "--retry", "4", "--retry-all-errors",
            "--output", package_path,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if download.returncode != 0:
        detail = download.stdout[-1200:].strip() or "no download output"
        raise RuntimeError("Could not download cloudflared: " + detail)
    install = subprocess.run(
        ["dpkg", "-i", package_path], text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False,
    )
    if install.returncode != 0 or not cloudflared_ready():
        detail = install.stdout[-1200:].strip() or "no installation output"
        raise RuntimeError("Could not install cloudflared: " + detail)


ensure_cloudflared()
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tunnel_lines = queue.Queue()


def collect_tunnel_output() -> None:
    assert tunnel.stdout is not None
    for line in tunnel.stdout:
        tunnel_lines.put(line)


threading.Thread(target=collect_tunnel_output, daemon=True).start()
public_url = ""
recent_tunnel_lines = []
deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS
while time.monotonic() < deadline and not public_url:
    if tunnel.poll() is not None:
        break
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    recent_tunnel_lines.append(line.rstrip())
    recent_tunnel_lines = recent_tunnel_lines[-10:]
    print(line, end="")
    match = re.search(r"https://[^\s\"']+\.trycloudflare\.com", line)
    if match:
        # The desktop Check Colab action is the authoritative public endpoint,
        # bearer-token, capability, and exact-model verification.
        public_url = match.group(0)

if not public_url:
    stop_process(tunnel)
    stop_process(worker)
    tail = "\n".join(recent_tunnel_lines) or "(no cloudflared output)"
    raise RuntimeError(
        f"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\n"
        "---- cloudflared output ----\n" + tail
    )

os.environ[URL_ENV] = public_url
os.environ[TOKEN_ENV] = TOKEN
os.environ[MODEL_ENV] = MODEL_ID
print("\nLA Studio exact-model Colab worker is ready")
print(URL_ENV + "=" + public_url)
print(TOKEN_ENV + "=" + TOKEN)
print(MODEL_ENV + "=" + MODEL_ID)
print("Click Check Colab in the matching LA Studio feature before running it.")
